# Data Collection and Pre-Processing Lab

**E-Commerce Data Engineering Road-map**

This notebook follows the required 12-step workflow: ingest → structure → profile → clean → transform → feature-engineer → aggregate → serialize. The primary transaction data is synthetic, while the secondary municipality metadata is a curated subset of the Government of Ontario open dataset.

## Step 1 — Hello, Data!
Load the raw transaction CSV and display the first three rows. Keeping the raw load separate makes it easy to compare later cleaning results with the original data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
raw_df = pd.read_csv(DATA_DIR / "transactions.csv")
raw_df.head(3)

,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,2025-10-09,C1063,Running Shoes,116.98,1,SAVE10,Hamilton
1,2026-07-13,C1023,Keyboard,65.40,1,FREESHIP,Mississauga
2,2025-08-12,C1060,Laptop,941.75,1,SAVE10,Toronto


## Step 2 — Pick the Right Container
A **dictionary** is useful for a transaction because field names such as `price` and `customer_id` act as meaningful keys. A **set** is appropriate for unique values such as shipping cities because it automatically removes duplicates. A class is used in the next step because it can keep transaction data and reusable methods such as `clean()` and `total()` together.

In [2]:
sample_transaction = raw_df.iloc[0].to_dict()
unique_cities = set(raw_df["shipping_city"].dropna())

print("Sample dictionary:", sample_transaction)
print("Unique cities:", unique_cities)

Sample dictionary: {'date': '2025-10-09', 'customer_id': 'C1063', 'product': 'Running Shoes', 'price': 116.98, 'quantity': 1, 'coupon_code': 'SAVE10', 'shipping_city': 'Hamilton'}
Unique cities: {'Brampton', 'Toronto', 'London', 'Waterloo', ' Toronto ', 'toronto', 'Mississauga', 'Hamilton', 'Kitchener', 'Ottawa'}


## Step 3 — Implement Functions and Data Structure
The `Transaction` class stores one transaction. `clean()` standardizes values and rejects impossible price/quantity values by converting them to missing values. `total()` calculates line revenue.

In [3]:
class Transaction:
    def __init__(self, date, customer_id, product, price, quantity, coupon_code, shipping_city):
        self.date = date
        self.customer_id = customer_id
        self.product = product
        self.price = price
        self.quantity = quantity
        self.coupon_code = coupon_code
        self.shipping_city = shipping_city

    def clean(self):
        if isinstance(self.customer_id, str):
            self.customer_id = self.customer_id.strip()
        if isinstance(self.product, str):
            self.product = self.product.strip()
        if isinstance(self.shipping_city, str):
            self.shipping_city = self.shipping_city.strip().title()
        if isinstance(self.coupon_code, str):
            self.coupon_code = self.coupon_code.strip().upper()
        if pd.notna(self.price) and self.price <= 0:
            self.price = np.nan
        if pd.notna(self.quantity) and self.quantity <= 0:
            self.quantity = np.nan
        return self

    def total(self):
        if pd.isna(self.price) or pd.isna(self.quantity):
            return np.nan
        return self.price * self.quantity

example = Transaction("2026-01-10", " C1001 ", " Laptop ", 900, 2, "save10", " toronto ")
example.clean()
print(example.__dict__)
print("Total:", example.total())

{'date': '2026-01-10', 'customer_id': 'C1001', 'product': 'Laptop', 'price': 900, 'quantity': 2, 'coupon_code': 'SAVE10', 'shipping_city': 'Toronto'}
Total: 1800


## Step 4 — Bulk Loaded
Convert the DataFrame into a list of dictionaries. This demonstrates how tabular data can be mapped into standard Python data structures for row-by-row processing.

In [4]:
transaction_records = raw_df.to_dict(orient="records")
print("Number of records:", len(transaction_records))
transaction_records[0]

Number of records: 503


{'date': '2025-10-09',
 'customer_id': 'C1063',
 'product': 'Running Shoes',
 'price': 116.98,
 'quantity': 1,
 'coupon_code': 'SAVE10',
 'shipping_city': 'Hamilton'}

## Step 5 — Quick Profiling
Profile the price field and use a set to count unique non-null shipping cities.

In [5]:
print("Minimum price:", raw_df["price"].min())
print("Mean price:", round(raw_df["price"].mean(), 2))
print("Maximum price:", raw_df["price"].max())

city_set = set(raw_df["shipping_city"].dropna())
print("Unique city values:", len(city_set))
print(city_set)

Minimum price: -25.0
Mean price: 169.46
Maximum price: 987.58
Unique city values: 10
{'Brampton', 'Toronto', 'London', 'Waterloo', ' Toronto ', 'toronto', 'Mississauga', 'Hamilton', 'Kitchener', 'Ottawa'}


## Step 6 — Spot the Grime
We check for missing values, duplicate rows, non-positive numeric values, invalid dates, and inconsistent city text. These checks reveal more than the required three dirty-data cases.

In [6]:
missing_before = raw_df.isna().sum()
duplicates_before = raw_df.duplicated().sum()
invalid_price_before = (raw_df["price"] <= 0).sum()
invalid_quantity_before = (raw_df["quantity"] <= 0).sum()
invalid_date_before = pd.to_datetime(raw_df["date"], errors="coerce").isna().sum()

print("Missing values before cleaning:")
print(missing_before)
print("Duplicate rows:", duplicates_before)
print("Non-positive prices:", invalid_price_before)
print("Non-positive quantities:", invalid_quantity_before)
print("Invalid dates:", invalid_date_before)
print("Raw city values:", sorted(raw_df["shipping_city"].dropna().unique()))

Missing values before cleaning:
date             0
customer_id      1
product          0
price            1
quantity         0
coupon_code      1
shipping_city    1
dtype: int64
Duplicate rows: 3
Non-positive prices: 1
Non-positive quantities: 1
Invalid dates: 1
Raw city values: [' Toronto ', 'Brampton', 'Hamilton', 'Kitchener', 'London', 'Mississauga', 'Ottawa', 'Toronto', 'Waterloo', 'toronto']


## Step 7 — Cleaning Rules
Cleaning is performed through the class `clean()` method for each row. Afterward, duplicate rows and records missing essential fields are removed. The before/after counts show the effect of the rules.

In [7]:
cleaned_records = []
for row in transaction_records:
    t = Transaction(**row).clean()
    cleaned_records.append(t.__dict__)

df = pd.DataFrame(cleaned_records)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

rows_before = len(df)
missing_essential_before = df[["date", "customer_id", "product", "price", "quantity", "shipping_city"]].isna().any(axis=1).sum()

df = df.drop_duplicates()
df = df.dropna(subset=["date", "customer_id", "product", "price", "quantity", "shipping_city"]).copy()
df["coupon_code"] = df["coupon_code"].fillna("NONE")

print("Rows before cleaning removals:", rows_before)
print("Duplicate rows before removal:", duplicates_before)
print("Rows with missing/invalid essential values:", missing_essential_before)
print("Rows after cleaning:", len(df))
print("Missing values after cleaning:")
print(df.isna().sum())
print("Clean city values:", sorted(df["shipping_city"].unique()))

Rows before cleaning removals: 503
Duplicate rows before removal: 3
Rows with missing/invalid essential values: 6
Rows after cleaning: 494
Missing values after cleaning:
date             0
customer_id      0
product          0
price            0
quantity         0
coupon_code      0
shipping_city    0
dtype: int64
Clean city values: ['Brampton', 'Hamilton', 'Kitchener', 'London', 'Mississauga', 'Ottawa', 'Toronto', 'Waterloo']


## Step 8 — Transformations
Coupon codes are converted into numeric discount percentages. We then calculate gross revenue, discount amount, and net revenue. `FREESHIP` affects shipping rather than item price, so its item discount is treated as 0%.

In [8]:
discount_map = {
    "SAVE10": 10,
    "WELCOME10": 10,
    "SAVE15": 15,
    "SAVE20": 20,
    "NONE": 0,
    "FREESHIP": 0
}

df["discount_percent"] = df["coupon_code"].map(discount_map).fillna(0)
df["gross_total"] = df["price"] * df["quantity"]
df["discount_amount"] = df["gross_total"] * (df["discount_percent"] / 100)
df["net_revenue"] = df["gross_total"] - df["discount_amount"]

df[["coupon_code", "discount_percent", "gross_total", "discount_amount", "net_revenue"]].head()

,coupon_code,discount_percent,gross_total,discount_amount,net_revenue
0,SAVE10,10,116.98,11.698,105.282
1,FREESHIP,0,65.40,0.000,65.400
2,SAVE10,10,941.75,94.175,847.575
3,FREESHIP,0,183.54,0.000,183.540
4,SAVE15,15,186.20,27.930,158.270


## Step 9 — Feature Engineering
Create `days_since_purchase` using the latest valid transaction date in the dataset as the reference date. Using a dataset-based reference keeps the notebook reproducible when the instructor runs it later. We also enrich the transactions using the secondary Ontario municipality metadata.

In [9]:
reference_date = df["date"].max()
df["days_since_purchase"] = (reference_date - df["date"]).dt.days

municipalities = pd.read_csv(DATA_DIR / "ontario_municipalities_subset.csv")
municipalities["municipality_name"] = municipalities["municipality_name"].str.strip().str.title()

df = df.merge(
    municipalities,
    how="left",
    left_on="shipping_city",
    right_on="municipality_name"
)

print("Reference date:", reference_date.date())
df[["date", "days_since_purchase", "shipping_city", "municipal_status", "municipal_structure"]].head()

Reference date: 2026-08-31


,date,days_since_purchase,shipping_city,municipal_status,municipal_structure
0,2025-10-09,326,Hamilton,City,Single-tier
1,2026-07-13,49,Mississauga,City,Lower-tier
2,2025-08-12,384,Toronto,City,Single-tier
3,2026-07-13,49,Hamilton,City,Single-tier
4,2025-01-07,601,Mississauga,City,Lower-tier


## Step 10 — Mini-Aggregation
Aggregate net revenue by shipping city. This provides a concise analytical result after cleaning and feature engineering.

In [10]:
revenue_by_city = (
    df.groupby("shipping_city", as_index=False)["net_revenue"]
      .sum()
      .sort_values("net_revenue", ascending=False)
)
revenue_by_city["net_revenue"] = revenue_by_city["net_revenue"].round(2)
revenue_by_city

,shipping_city,net_revenue
3,London,32664.11
1,Hamilton,30811.34
7,Waterloo,30579.81
6,Toronto,28433.42
5,Ottawa,26160.63
2,Kitchener,24703.59
4,Mississauga,23369.90
0,Brampton,21820.51


In [11]:
top_city = revenue_by_city.iloc[0]
print(f"Insight: {top_city['shipping_city']} generated the highest net revenue in this dataset at ${top_city['net_revenue']:,.2f}.")

Insight: London generated the highest net revenue in this dataset at $32,664.11.


## Step 11 — Serialization Checkpoint
Serialize the cleaned and enriched dataset in **two formats: CSV and JSON**, satisfying the serialization learning objective.

In [12]:
output_csv = DATA_DIR / "cleaned_transactions.csv"
output_json = DATA_DIR / "cleaned_transactions.json"

df.to_csv(output_csv, index=False)
df.to_json(output_json, orient="records", indent=2, date_format="iso")

print("Saved:", output_csv)
print("Saved:", output_json)

Saved: ../data/cleaned_transactions.csv
Saved: ../data/cleaned_transactions.json


## Step 12 — Soft Interview Reflection
Functions and methods made the workflow easier to organize and reuse. Instead of repeating the same cleaning instructions for every transaction, I placed the rules inside the `clean()` method and applied them consistently. The `total()` method also keeps the revenue calculation close to the transaction data it belongs to. This approach makes the code easier to read, test, and update because a rule only needs to be changed in one place. In a larger data-engineering pipeline, reusable functions reduce duplication and help ensure that the same processing logic is applied whenever new data arrives.

## Data Dictionary
The dictionary combines fields from the synthetic primary transaction CSV, the Government of Ontario municipality metadata subset, and fields engineered in this notebook. The municipality source is the Ontario Data Catalogue's **Municipalities** dataset.

| Field | Type | Description | Source |
|---|---|---|---|
| date | datetime | Date of the purchase transaction | Primary synthetic CSV |
| customer_id | string | Synthetic identifier for the customer | Primary synthetic CSV |
| product | string | Product purchased | Primary synthetic CSV |
| price | float | Unit selling price before coupon discount | Primary synthetic CSV |
| quantity | float/int | Number of units purchased | Primary synthetic CSV |
| coupon_code | string | Promotion/coupon attached to the transaction | Primary synthetic CSV |
| shipping_city | string | Destination city, standardized during cleaning | Primary synthetic CSV + cleaning |
| discount_percent | numeric | Percentage parsed from the coupon mapping; non-item discounts map to 0 | Created in Step 8 |
| gross_total | float | `price × quantity` before discount | Created in Step 8 |
| discount_amount | float | `gross_total × discount_percent / 100` | Created in Step 8 |
| net_revenue | float | Gross total minus discount amount | Created in Step 8 |
| days_since_purchase | integer | Days between purchase and latest valid date in this dataset | Created in Step 9 |
| municipality_name | string | Municipality name used to match shipping city | Ontario Municipalities open data |
| municipal_status | string | Municipal status/category for the matched city | Ontario Municipalities open data |
| municipal_structure | string | Simplified municipal structure used in the curated subset | Secondary metadata subset |